<a href="https://colab.research.google.com/github/Damainx22/RGV-Business-Survival/blob/main/notebooks/05_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# Notebook 05: Merge All Datasets
# Purpose: Load all 4 cleaned datasets and merge
#          them into one master DataFrame for modeling.
#          Joins SBA loans with ACS demographics,
#          CBP business density, and BDS firm dynamics
#          on zip code and year.
# ============================================

# Mount Google Drive so we can access our cleaned data files
from google.colab import drive
drive.mount('/content/drive')

# Standard imports
import os
import pandas as pd

# Define folder paths
RAW = '/content/drive/MyDrive/rgv_business_survival/data/raw'
CLEANED = '/content/drive/MyDrive/rgv_business_survival/data/cleaned'
MERGED = '/content/drive/MyDrive/rgv_business_survival/data/merged'

# Make sure merged folder exists
os.makedirs(MERGED, exist_ok=True)

# Verify cleaned files are accessible
print("Cleaned files:", sorted(os.listdir(CLEANED)))

Mounted at /content/drive
Cleaned files: ['acs_rgv_clean.csv', 'acs_texas_clean.csv', 'bds_texas_clean.csv', 'cbp_texas.csv', 'sba_rgv_clean.csv', 'sba_texas_clean.csv']


In [3]:
# Load all 4 cleaned datasets into separate DataFrames
# Each file was cleaned and filtered in notebooks 01-04

# SBA loan data — 8,119 Texas loans FY2018-2022 with default outcome
sba = pd.read_csv(f'{CLEANED}/sba_texas_clean.csv', low_memory=False)

# ACS demographics — median income, poverty rate, unemployment rate by zip code and year
acs = pd.read_csv(f'{CLEANED}/acs_texas_clean.csv', low_memory=False)

# CBP business patterns — total establishment count by zip code and year
cbp = pd.read_csv(f'{CLEANED}/cbp_texas.csv', low_memory=False)

# BDS business dynamics — firm birth/death rates by county and year
bds = pd.read_csv(f'{CLEANED}/bds_texas_clean.csv', low_memory=False)

# Verify shapes before merging
print(f'SBA loans: {sba.shape}')
print(f'ACS demographics: {acs.shape}')
print(f'CBP business patterns: {cbp.shape}')
print(f'BDS firm dynamics: {bds.shape}')

SBA loans: (8119, 16)
ACS demographics: (9783, 11)
CBP business patterns: (10958, 3)
BDS firm dynamics: (1275, 9)


In [5]:
# ============================================
# Step 1: Merge SBA loans with ACS demographics
# Join on zip code + year so each loan gets the
# demographic context of its zip code in that year
# ============================================
model_df = pd.merge(sba, acs,
                    left_on=['borrzip', 'approvalfiscalyear'],
                    right_on=['zip_code', 'year'])

print(f'After SBA + ACS merge: {model_df.shape}')

# ============================================
# Step 2: Merge with CBP business density
# Join on zip code + year to attach total
# establishment count for that zip code
# ============================================
model_df = pd.merge(model_df, cbp,
                    left_on=['borrzip', 'approvalfiscalyear'],
                    right_on=['zip', 'year'])

print(f'After + CBP merge: {model_df.shape}')

# ============================================
# Step 3: Merge with BDS firm dynamics
# BDS is at county level not zip level
# We need to match on county FIPS code + year
# First check what county columns look like
# ============================================
print("SBA counties sample:", sba['projectcounty'].value_counts().head(5))
print("BDS county codes sample:", bds['cty'].unique()[:10])

After SBA + ACS merge: (8064, 27)
After + CBP merge: (8064, 30)
SBA counties sample: projectcounty
HARRIS     1474
DALLAS      965
TARRANT     676
COLLIN      527
TRAVIS      515
Name: count, dtype: int64
BDS county codes sample: [ 1  3  5  7  9 11 13 15 17 19]


In [8]:
# Drop duplicate and unnecessary columns
model_df = model_df.drop(columns=['zip_code', 'zip', 'year_y', 'state', 'NAME', 'poverty_count', 'unemployed_count', 'labor_force_count'])

# Fill remaining nulls with median
model_df['poverty_rate'] = model_df['poverty_rate'].fillna(model_df['poverty_rate'].median())
model_df['unemployment_rate'] = model_df['unemployment_rate'].fillna(model_df['unemployment_rate'].median())

# Rename year_x to year for clarity
model_df = model_df.rename(columns={'year_x': 'year'})

print(f'Final shape: {model_df.shape}')
print(model_df.isnull().sum())

Final shape: (8064, 22)
borrname                 0
borrcity                 0
borrzip                  0
grossapproval            0
sbaguaranteedapproval    0
approvalfiscalyear       0
initialinterestrate      0
terminmonths             0
naicscode                0
naicsdescription         0
businesstype             0
businessage              0
loanstatus               0
defaulted                0
jobssupported            0
projectcounty            0
median_income            0
total_population         0
poverty_rate             0
unemployment_rate        0
year                     0
est                      0
dtype: int64


In [9]:
# Save the merged dataset to data/merged/ folder
model_df.to_csv(f'{MERGED}/model_df.csv', index=False)
print(f'Saved: {len(model_df)} rows, {len(model_df.columns)} columns')
print(os.listdir(MERGED))

Saved: 8064 rows, 22 columns
['model_df.csv']


In [10]:
print(os.listdir(MERGED))

['model_df.csv']
